In [ ]:
# Ferry Ridership

In [ ]:
import pandas as pd
import plotly.express as px
import h3
import numpy as np

In [ ]:
# Ridership Export
ferry_data_path = 'data/processed/domain/human_layer/ferry/wsf_effort_hourly_2020_2026.parquet'
ferry_data_path = 'data/processed/domain/human_layer/ferry/wsf_ridership.parquet'

In [ ]:
ridership = pd.read_parquet(ferry_data_path)

ridership['month'] = ridership['calendar_date'].dt.month
ridership['year'] = ridership['calendar_date'].dt.year

columns_to_use = [
    'calendar_date', 'departure_hour_local', 'year', 'month', 
    'origin_terminal', 'destination_terminal', 'route_group', 'segment_id', 
    'reported_total_riders'
    ]

rider_df = ridership[columns_to_use]

## Ridership by Year

In [ ]:
rider_yr_counts = rider_df.groupby('year', as_index = False)['reported_total_riders'].sum()
fig = px.bar(rider_yr_counts, x = 'year', y = 'reported_total_riders', title = 'Yearly Total Riders')
fig.show()

## Ridership by Month

In [ ]:
rider_moyr_counts = rider_df.groupby(['year', 'month'], as_index = False)['reported_total_riders'].sum()
rider_moyr_counts['month_year'] = rider_moyr_counts.apply(lambda x: f"{x['year']}-{x['month']}-01", axis = 1)

fig = px.bar(rider_moyr_counts, x = 'month_year', y = 'reported_total_riders', title = 'Monthly Total Riders')
fig.show()

## Ridership in June 2026

In [ ]:
rider_ju26_counts = (
    rider_df[
        (rider_df["year"] == 2026) &
        (rider_df["month"] == 6)
    ]
    .groupby("calendar_date", as_index=False)["reported_total_riders"]
    .sum()
)

# Ensure Plotly and pandas treat this as a date
rider_ju26_counts["calendar_date"] = pd.to_datetime(
    rider_ju26_counts["calendar_date"]
)

# Monday=0, ..., Saturday=5, Sunday=6
rider_ju26_counts["day_name"] = (
    rider_ju26_counts["calendar_date"].dt.day_name()
)

rider_ju26_counts["day_type"] = np.where(
    rider_ju26_counts["calendar_date"].dt.dayofweek >= 5,
    "Weekend",
    "Weekday",
)

fig = px.bar(
    rider_ju26_counts,
    x="calendar_date",
    y="reported_total_riders",
    color="day_type",
    category_orders={"day_type": ["Weekday", "Weekend"]},
    color_discrete_map={
        "Weekday": "#4C78A8",
        "Weekend": "#F58518",
    },
    hover_data={
        "calendar_date": "|%A, %B %d, %Y",
        "day_name": True,
        "day_type": True,
        "reported_total_riders": ":,.0f",
    },
    title="Daily Total Riders, June 2026",
    labels={
        "calendar_date": "Date",
        "reported_total_riders": "Total riders",
        "day_type": "Day type",
    },
)

fig.update_layout(
    xaxis_tickformat="%b %d",
    bargap=0.15,
)

fig.show()

In [ ]:
rider_ju26_hourly_counts = (
    rider_df[
        (rider_df["year"] == 2026) &
        (rider_df["month"] == 6)
    ]
    .groupby(["calendar_date", "departure_hour_local"], as_index=False)["reported_total_riders"]
    .sum()
)

In [ ]:
fig = px.box(rider_ju26_hourly_counts, x = 'departure_hour_local', y = 'reported_total_riders', title = 'Hourly Distribution')
fig.show()

## Ridership by Route

In [ ]:
rider_df.groupby(['route_group'], as_index = False)['reported_total_riders'].sum()

In [ ]:
rider_df_route_my = rider_df.groupby(['route_group', 'year', 'month'], as_index = False)['reported_total_riders'].sum()
rider_df_route_my['month_year'] = rider_df_route_my.apply(lambda x: f"{x['year']}-{x['month']}-01", axis = 1)

In [ ]:
rider_df_route_my_pvt = pd.pivot_table(
    rider_df_route_my, index = 'route_group', columns = 'month_year', values = 'reported_total_riders').fillna(0)


fig = px.imshow(rider_df_route_my_pvt)#, x = '')
fig.show()

# British Columbia Ferries

BC Ferries publishes official **monthly route-level passenger and vehicle totals**, not daily or sailing-level ridership. The daily and hourly products below allocate each published monthly route total using the observed WSF systemwide month × weekday × hour profile. Consequently:

- monthly, yearly, and route totals are published BC Ferries values;
- daily and hourly charts are modeled temporal allocations, not observed BC sailings; and
- every monthly route total is conserved exactly and retains its source-report URL and hash.

In [ ]:
import json
from pathlib import Path


def find_ferry_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "notebooks/data/ferry_utils").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the OrcaCast repository.")


FERRY_REPO_ROOT = find_ferry_repo_root()
FERRY_UTILS_DIR = FERRY_REPO_ROOT / "notebooks/data/ferry_utils"
BC_REPORT_CACHE_DIR = FERRY_UTILS_DIR / "data/raw/bc_ferries_traffic"
BC_FERRY_DATA = FERRY_UTILS_DIR / "bc_ferries_estimated_ridership.parquet"
BC_FERRY_METADATA = BC_FERRY_DATA.with_suffix(BC_FERRY_DATA.suffix + ".metadata.json")
BC_OUTPUT_DIR = FERRY_UTILS_DIR / "outputs/bc_ferries"
BC_REPORT_PREVIEW_DIR = BC_REPORT_CACHE_DIR / "previews"
BC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not BC_FERRY_DATA.exists():
    raise FileNotFoundError(
        f"Missing {BC_FERRY_DATA}. Run wsf_ridership_to_parquet.py --source bc first; "
        f"reports will be cached in {BC_REPORT_CACHE_DIR}."
    )

bc_ridership = pd.read_parquet(BC_FERRY_DATA)
bc_metadata = json.loads(BC_FERRY_METADATA.read_text(encoding="utf-8"))
required_bc_columns = {
    "service_date", "report_month", "departure_hour_local", "route_number",
    "route_name", "reported_total_riders", "vehicles",
    "published_monthly_passengers", "published_monthly_vehicles",
    "is_estimated", "source_report_url",
}
missing_bc_columns = required_bc_columns - set(bc_ridership.columns)
if missing_bc_columns:
    raise ValueError(f"BC ferry data is missing columns: {sorted(missing_bc_columns)}")
if not bc_ridership["is_estimated"].all():
    raise ValueError("Expected every BC route-hour row to be explicitly marked as estimated.")

bc_ridership["service_date"] = pd.to_datetime(bc_ridership["service_date"])
bc_ridership["report_month"] = pd.to_datetime(bc_ridership["report_month"])
bc_ridership["year"] = bc_ridership["report_month"].dt.year
bc_ridership["month"] = bc_ridership["report_month"].dt.month

# Published totals are repeated on route-hour rows, so take one value per route-month.
bc_published = (
    bc_ridership.groupby(["report_month", "route_number", "route_name"], as_index=False)
    .agg(
        published_passengers=("published_monthly_passengers", "first"),
        published_vehicles=("published_monthly_vehicles", "first"),
        source_report_url=("source_report_url", "first"),
    )
)
bc_published["year"] = bc_published["report_month"].dt.year
bc_published["month"] = bc_published["report_month"].dt.month

# Recheck the artifact contract inside the notebook instead of trusting metadata alone.
bc_allocated = (
    bc_ridership.groupby(["report_month", "route_number"], as_index=False)
    .agg(allocated_passengers=("reported_total_riders", "sum"), allocated_vehicles=("vehicles", "sum"))
)
bc_conservation = bc_published.merge(
    bc_allocated, on=["report_month", "route_number"], how="left", validate="one_to_one"
)
passenger_error = (bc_conservation["allocated_passengers"] - bc_conservation["published_passengers"]).abs()
vehicle_mask = bc_conservation["published_vehicles"].notna()
vehicle_error = (
    bc_conservation.loc[vehicle_mask, "allocated_vehicles"]
    - bc_conservation.loc[vehicle_mask, "published_vehicles"]
).abs()
if passenger_error.max() > 1e-6 or vehicle_error.max() > 1e-6:
    raise ValueError("BC temporal allocations do not conserve their published route-month totals.")

bc_summary = pd.DataFrame(
    {
        "measure": ["Coverage", "Routes", "Published passengers", "Published vehicles", "Temporal rows", "Estimation method"],
        "value": [
            f"{bc_ridership['service_date'].min():%Y-%m-%d} through {bc_ridership['service_date'].max():%Y-%m-%d}",
            f"{bc_published['route_number'].nunique():,}",
            f"{bc_published['published_passengers'].sum():,.0f}",
            f"{bc_published['published_vehicles'].sum():,.0f}",
            f"{len(bc_ridership):,}",
            bc_metadata["estimation_method"],
        ],
    }
)
display(bc_summary)
bc_reports = pd.DataFrame(bc_metadata["source_reports"])
display(bc_reports)
print(
    "Monthly-total conservation validated independently; "
    f"maximum passenger error={passenger_error.max():.3g}, vehicle error={vehicle_error.max():.3g}."
)

## BC Ferries ridership by year

In [ ]:
bc_yearly = (
    bc_published.groupby("year", as_index=False)["published_passengers"]
    .sum()
)
fig = px.bar(
    bc_yearly,
    x="year",
    y="published_passengers",
    title="BC Ferries published passenger totals by available year",
    labels={"year": "Year", "published_passengers": "Published passengers"},
)
fig.update_traces(hovertemplate="Year %{x}<br>Published passengers: %{y:,.0f}<extra></extra>")
fig.write_html(BC_OUTPUT_DIR / "bc_ferries_yearly_passengers.html", include_plotlyjs=True)
fig.show()

## BC Ferries passengers and vehicles by month

In [ ]:
bc_monthly = (
    bc_published.groupby("report_month", as_index=False)
    .agg(
        published_passengers=("published_passengers", "sum"),
        published_vehicles=("published_vehicles", lambda values: values.sum(min_count=1)),
    )
)
bc_monthly_long = bc_monthly.melt(
    id_vars="report_month",
    value_vars=["published_passengers", "published_vehicles"],
    var_name="measure",
    value_name="published_total",
)
bc_monthly_long["measure"] = bc_monthly_long["measure"].map(
    {"published_passengers": "Passengers", "published_vehicles": "Vehicles"}
)
fig = px.bar(
    bc_monthly_long,
    x="report_month",
    y="published_total",
    facet_row="measure",
    color="measure",
    title="BC Ferries published monthly traffic totals",
    labels={"report_month": "Report month", "published_total": "Published total"},
)
fig.update_yaxes(matches=None)
fig.update_xaxes(tickformat="%b %Y")
fig.update_traces(hovertemplate="%{x|%B %Y}<br>Total: %{y:,.0f}<extra></extra>")
fig.write_html(BC_OUTPUT_DIR / "bc_ferries_monthly_traffic.html", include_plotlyjs=True)
bc_monthly.to_csv(BC_OUTPUT_DIR / "bc_ferries_monthly_traffic.csv", index=False)
fig.show()

## Estimated BC Ferries ridership in June 2026

These daily and hourly values are the WSF-profile allocation of BC Ferries' published June route totals. They should be interpreted as exposure estimates, not actual BC sailing counts.

In [ ]:
bc_june_2026 = bc_ridership.loc[
    (bc_ridership["year"] == 2026) & (bc_ridership["month"] == 6)
].copy()
if bc_june_2026.empty:
    raise ValueError("The BC estimate does not contain June 2026.")

bc_june_daily = (
    bc_june_2026.groupby("service_date", as_index=False)["reported_total_riders"]
    .sum()
)
bc_june_daily["day_name"] = bc_june_daily["service_date"].dt.day_name()
bc_june_daily["day_type"] = np.where(
    bc_june_daily["service_date"].dt.dayofweek >= 5, "Weekend", "Weekday"
)
fig = px.bar(
    bc_june_daily,
    x="service_date",
    y="reported_total_riders",
    color="day_type",
    category_orders={"day_type": ["Weekday", "Weekend"]},
    color_discrete_map={"Weekday": "#4E79A7", "Weekend": "#F28E2B"},
    hover_data={"day_name": True, "reported_total_riders": ":,.0f"},
    title="Estimated daily BC Ferries passengers, June 2026",
    labels={"service_date": "Date", "reported_total_riders": "Estimated passengers", "day_type": "Day type"},
)
fig.update_xaxes(tickformat="%b %d")
fig.write_html(BC_OUTPUT_DIR / "bc_ferries_june_2026_daily_estimate.html", include_plotlyjs=True)
fig.show()

In [ ]:
bc_june_hourly = (
    bc_june_2026.groupby(["service_date", "departure_hour_local"], as_index=False)
    ["reported_total_riders"].sum()
)
fig = px.box(
    bc_june_hourly,
    x="departure_hour_local",
    y="reported_total_riders",
    points=False,
    title="Estimated BC Ferries hourly passenger distribution, June 2026",
    labels={"departure_hour_local": "Local hour", "reported_total_riders": "Estimated passengers"},
)
fig.write_html(BC_OUTPUT_DIR / "bc_ferries_june_2026_hourly_estimate.html", include_plotlyjs=True)
fig.show()

## BC Ferries ridership by route

In [ ]:
bc_route_totals = (
    bc_published.groupby(["route_number", "route_name"], as_index=False)
    .agg(
        published_passengers=("published_passengers", "sum"),
        published_vehicles=("published_vehicles", lambda values: values.sum(min_count=1)),
        months_reported=("report_month", "nunique"),
    )
    .sort_values("published_passengers", ascending=False)
)
display(bc_route_totals)
bc_route_totals.to_csv(BC_OUTPUT_DIR / "bc_ferries_route_totals.csv", index=False)
fig = px.bar(
    bc_route_totals.sort_values("published_passengers"),
    x="published_passengers",
    y="route_name",
    orientation="h",
    hover_data={"route_number": True, "published_vehicles": ":,.0f", "months_reported": True},
    title="BC Ferries published passengers by route, available months",
    labels={"published_passengers": "Published passengers", "route_name": "Route"},
    height=750,
)
fig.write_html(BC_OUTPUT_DIR / "bc_ferries_route_totals.html", include_plotlyjs=True)
fig.show()

In [ ]:
bc_route_month = bc_published.pivot_table(
    index="route_name",
    columns="report_month",
    values="published_passengers",
    aggfunc="sum",
    fill_value=0,
)
bc_route_month.columns = [timestamp.strftime("%Y-%m") for timestamp in bc_route_month.columns]
fig = px.imshow(
    bc_route_month,
    aspect="auto",
    color_continuous_scale="Blues",
    title="BC Ferries published passengers by route and report month",
    labels={"x": "Report month", "y": "Route", "color": "Passengers"},
)
fig.update_traces(hovertemplate="Route: %{y}<br>Month: %{x}<br>Passengers: %{z:,.0f}<extra></extra>")
fig.update_layout(height=750)
fig.write_html(BC_OUTPUT_DIR / "bc_ferries_route_month_heatmap.html", include_plotlyjs=True)
fig.show()
print(f"Saved BC Ferry tables and interactive charts to {BC_OUTPUT_DIR}")

## Washington and British Columbia ferry route segments

Download the official WSDOT ferry-route layer and the Government of British Columbia Coastal Ferry Routes layer, retain coastal routes relevant to the SRKW and transient study domains, explode multipart routes into LineString segments, and save reusable GeoJSON/Parquet artifacts.

In [ ]:
from pathlib import Path

import geopandas as gpd
import plotly.graph_objects as go
import requests
import yaml
from pyproj import Geod
from shapely.geometry import box


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "config/data/sightings.yaml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the OrcaCast repository.")


def read_arcgis_geojson(layer_url: str) -> gpd.GeoDataFrame:
    response = requests.get(
        f"{layer_url}/query",
        params={
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "outSR": 4326,
            "f": "geojson",
        },
        headers={"User-Agent": "OrcaCast ferry-route research/1.0"},
        timeout=(20, 180),
    )
    response.raise_for_status()
    payload = response.json()
    if payload.get("type") != "FeatureCollection":
        raise RuntimeError(f"ArcGIS response was not GeoJSON: {payload}")
    return gpd.GeoDataFrame.from_features(payload["features"], crs="EPSG:4326")


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "notebooks/data/ferry_utils/outputs/ferry_routes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ROUTES_GEOJSON = OUTPUT_DIR / "wa_bc_ferry_route_segments.geojson"
ROUTES_PARQUET = OUTPUT_DIR / "wa_bc_ferry_route_segments.parquet"
ROUTES_MAP = OUTPUT_DIR / "wa_bc_ferry_route_segments_map.html"

WSDOT_ROUTES_URL = (
    "https://data.wsdot.wa.gov/arcgis/rest/services/Shared/"
    "FerryRoutes/FeatureServer/1"
)
BC_COASTAL_ROUTES_URL = (
    "https://delivery.maps.gov.bc.ca/arcgis/rest/services/mpcm/"
    "bcgwpub/MapServer/211"
)

wa_raw = read_arcgis_geojson(WSDOT_ROUTES_URL)
bc_raw = read_arcgis_geojson(BC_COASTAL_ROUTES_URL)

with (REPO_ROOT / "config/data/sightings.yaml").open(encoding="utf-8") as stream:
    sightings_config = yaml.safe_load(stream)
srkw_bounds = sightings_config["model_ranges"]["SRKW"]
srkw_geometry = box(
    srkw_bounds["min_lon"],
    srkw_bounds["min_lat"],
    srkw_bounds["max_lon"],
    srkw_bounds["max_lat"],
)
transient_bounds = sightings_config["model_ranges"]["TRANSIENT"]
transient_geometry = box(
    transient_bounds["min_lon"],
    transient_bounds["min_lat"],
    transient_bounds["max_lon"],
    transient_bounds["max_lat"],
)

# The WSDOT service also contains inland-lake routes. Intersecting it with
# the configured SRKW model range keeps the marine Washington routes. The BC layer
# is already explicitly coastal; the transient bbox limits it to model scope.
wa = wa_raw.loc[
    wa_raw["Display"].fillna("").str.strip().ne("")
    & wa_raw.intersects(srkw_geometry)
].copy()
bc = bc_raw.loc[bc_raw.intersects(transient_geometry)].copy()

wa_routes = gpd.GeoDataFrame(
    {
        "route_id": "WA-" + wa["OBJECTID"].astype(int).astype(str).str.zfill(4),
        "route_name": wa["Display"].str.strip(),
        "operator": wa["Owner"].fillna("Unknown"),
        "jurisdiction": "Washington",
        "route_number": wa["SR"].fillna(""),
        "service_class": "Not specified",
        "frequency": "Not specified",
        "source_dataset": "WSDOT Ferry Routes (Public & Private)",
        "source_url": WSDOT_ROUTES_URL,
        "geometry": wa.geometry,
    },
    geometry="geometry",
    crs="EPSG:4326",
)
bc_routes = gpd.GeoDataFrame(
    {
        "route_id": "BC-" + bc["FERRY_ROUTE_ID"].astype(int).astype(str).str.zfill(4),
        "route_name": bc["ROUTE_NAME"].fillna("Unnamed route"),
        "operator": bc["ROUTE_OPERATOR"].fillna("Unknown"),
        "jurisdiction": "British Columbia",
        "route_number": "",
        "service_class": bc["MANIFEST_TYPE"].fillna("Not specified"),
        "frequency": bc["FREQUENCY_OF_USE_IND"].fillna("Not specified"),
        "source_dataset": "Government of BC Coastal Ferry Routes",
        "source_url": BC_COASTAL_ROUTES_URL,
        "geometry": bc.geometry,
    },
    geometry="geometry",
    crs="EPSG:4326",
)

routes = gpd.GeoDataFrame(
    pd.concat([wa_routes, bc_routes], ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326",
)
routes["overlaps_srkw_domain"] = routes.intersects(srkw_geometry)
routes["overlaps_transient_domain"] = routes.intersects(transient_geometry)
routes["domain_scope"] = np.select(
    [
        routes["overlaps_srkw_domain"] & routes["overlaps_transient_domain"],
        routes["overlaps_srkw_domain"],
        routes["overlaps_transient_domain"],
    ],
    ["SRKW + transient", "SRKW", "transient"],
    default="outside configured domains",
)

route_segments = routes.explode(ignore_index=True)
route_segments = route_segments.loc[
    route_segments.geometry.notna()
    & ~route_segments.geometry.is_empty
    & route_segments.geom_type.eq("LineString")
].copy()
route_segments["segment_number"] = (
    route_segments.groupby("route_id").cumcount() + 1
)
route_segments["segment_id"] = (
    route_segments["route_id"]
    + "-S"
    + route_segments["segment_number"].astype(str).str.zfill(2)
)
geod = Geod(ellps="WGS84")
route_segments["segment_length_km"] = route_segments.geometry.map(
    lambda geometry: abs(geod.geometry_length(geometry)) / 1_000
).round(2)
route_segments = route_segments.sort_values(
    ["jurisdiction", "route_name", "segment_number"]
).reset_index(drop=True)

route_segments.to_file(ROUTES_GEOJSON, driver="GeoJSON")
route_segments.to_parquet(ROUTES_PARQUET, index=False)

route_summary = (
    route_segments.groupby("jurisdiction", as_index=False)
    .agg(routes=("route_id", "nunique"), line_segments=("segment_id", "count"))
)
display(route_summary)
print(f"Saved {len(route_segments):,} LineString segments to {ROUTES_GEOJSON}")
print(f"Saved analysis-ready Parquet to {ROUTES_PARQUET}")

route_colors = {"Washington": "#F28E2B", "British Columbia": "#4E79A7"}
fig = go.Figure()
legend_seen: set[str] = set()
for segment in route_segments.itertuples(index=False):
    coordinates = list(segment.geometry.coords)
    hover = (
        f"<b>{segment.route_name}</b><br>"
        f"Operator: {segment.operator}<br>"
        f"Jurisdiction: {segment.jurisdiction}<br>"
        f"Service: {segment.service_class}<br>"
        f"Frequency: {segment.frequency}<br>"
        f"Domain overlap: {segment.domain_scope}<br>"
        f"Segment: {segment.segment_id}<br>"
        f"Length: {segment.segment_length_km:,.1f} km<br>"
        f"Source: {segment.source_dataset}"
    )
    fig.add_trace(
        go.Scattermap(
            lon=[coordinate[0] for coordinate in coordinates],
            lat=[coordinate[1] for coordinate in coordinates],
            mode="lines",
            line={"width": 2.5, "color": route_colors[segment.jurisdiction]},
            opacity=0.85,
            name=segment.jurisdiction,
            legendgroup=segment.jurisdiction,
            showlegend=segment.jurisdiction not in legend_seen,
            hovertext=[hover] * len(coordinates),
            hovertemplate="%{hovertext}<extra></extra>",
        )
    )
    legend_seen.add(segment.jurisdiction)

route_bounds = route_segments.total_bounds
fig.update_layout(
    title="Washington and British Columbia coastal ferry routes",
    map={
        "style": "open-street-map",
        "center": {
            "lon": float((route_bounds[0] + route_bounds[2]) / 2),
            "lat": float((route_bounds[1] + route_bounds[3]) / 2),
        },
        "zoom": 4.1,
    },
    height=800,
    margin={"l": 0, "r": 0, "t": 55, "b": 0},
    legend={"orientation": "h", "y": 0.99, "x": 0.01},
    hoverlabel={"align": "left"},
)
fig.write_html(ROUTES_MAP, include_plotlyjs=True, full_html=True)
print(f"Saved interactive hover map to {ROUTES_MAP}")
fig.show()

# Kingston–Edmonds voyage footprint and H3 time-in-grid prototype

This diagnostic stops after selecting and displaying the route H3 cells. It uses the latest complete ridership date, a representative Kingston → Edmonds sailing nearest noon, the project human/AIS resolution (H3 r7), WSDOT's approximate 30-minute crossing time, and the selected vessel's beam as the moving circular footprint diameter.

The centerline distance intersecting each H3 cell determines its voyage weight. The swept vessel-width corridor selects touched cells; cells touched only by the footprint edge remain visible with zero centerline weight. Daily ridership is allocated across the route cells without changing the date's total.

In [ ]:
from shapely.geometry import LineString, Point, Polygon

# Editable prototype parameters.
KE_H3_RESOLUTION = 7
KE_VOYAGE_DURATION_MIN = 30.0
KE_SELECTED_ORIGIN = "Kingston"
KE_SELECTED_DESTINATION = "Edmonds"
KE_TARGET_DEPARTURE_CLOCK = pd.Timedelta(hours=12)
KE_VESSEL_BEAM_FT = {"Spokane": 87.0, "Walla Walla": 87.0, "Puyallup": 90.0}
KE_OUTPUT_DIR = REPO_ROOT / "notebooks/data/ferry_utils/outputs/kingston_edmonds_h3"
KE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ke_route_matches = route_segments.loc[
    route_segments["jurisdiction"].eq("Washington")
    & route_segments["route_name"].str.contains("Edmonds.*Kingston|Kingston.*Edmonds", case=False, regex=True)
].copy()
if len(ke_route_matches) != 1:
    raise ValueError(f"Expected one Kingston–Edmonds LineString; found {len(ke_route_matches)}.")
ke_route = ke_route_matches.iloc[0]

ke_ridership = ridership.loc[
    ridership["route_group"].eq("Edmonds-Kingston")
    & ridership["origin_terminal"].isin(["Kingston", "Edmonds"])
    & ridership["destination_terminal"].isin(["Kingston", "Edmonds"])
].copy()
if ke_ridership.empty:
    raise ValueError("No Edmonds–Kingston sailing-level ridership rows were found.")
KE_SELECTED_DATE = ke_ridership["service_date"].max().normalize()
ke_selected_direction = ke_ridership.loc[
    ke_ridership["service_date"].eq(KE_SELECTED_DATE)
    & ke_ridership["origin_terminal"].eq(KE_SELECTED_ORIGIN)
    & ke_ridership["destination_terminal"].eq(KE_SELECTED_DESTINATION)
].copy()
target_departure = KE_SELECTED_DATE + KE_TARGET_DEPARTURE_CLOCK
ke_selected_direction["target_delta"] = (ke_selected_direction["departure_local"] - target_departure).abs()
selected_voyage = ke_selected_direction.sort_values("target_delta").iloc[0]

# The WSDOT geometry is stored Kingston (west) to Edmonds (east). Orient explicitly.
route_coordinates = list(ke_route.geometry.coords)
if KE_SELECTED_ORIGIN == "Kingston" and route_coordinates[0][0] > route_coordinates[-1][0]:
    route_coordinates.reverse()
elif KE_SELECTED_ORIGIN == "Edmonds" and route_coordinates[0][0] < route_coordinates[-1][0]:
    route_coordinates.reverse()
ke_oriented_wgs84 = LineString(route_coordinates)
selected_beam_ft = KE_VESSEL_BEAM_FT.get(str(selected_voyage["vessel_name"]), 90.0)
selected_beam_m = selected_beam_ft * 0.3048

voyage_summary = pd.DataFrame(
    {
        "attribute": ["Route feature", "Route segment", "Selected date", "Direction", "Departure", "Vessel", "Reported riders", "Approx. duration", "Vessel beam", "H3 resolution"],
        "value": [
            f"{ke_route.route_id}: {ke_route.route_name}", ke_route.segment_id,
            f"{KE_SELECTED_DATE:%Y-%m-%d}", f"{KE_SELECTED_ORIGIN} → {KE_SELECTED_DESTINATION}",
            f"{selected_voyage['departure_local']:%Y-%m-%d %H:%M}", selected_voyage["vessel_name"],
            f"{selected_voyage['reported_total_riders']:,.0f}", f"{KE_VOYAGE_DURATION_MIN:.0f} minutes",
            f"{selected_beam_ft:.0f} ft ({selected_beam_m:.1f} m)", KE_H3_RESOLUTION,
        ],
    }
)
display(voyage_summary)
display(selected_voyage[["sailing_id", "service_date", "departure_local", "origin_terminal", "destination_terminal", "vessel_name", "reported_total_riders"]].to_frame().T)

In [ ]:
# Project to UTM 10N so route, footprint, and cell-intersection lengths are metric.
ke_route_metric_gdf = gpd.GeoDataFrame(geometry=[ke_oriented_wgs84], crs=4326).to_crs(32610)
ke_route_metric = ke_route_metric_gdf.geometry.iloc[0]
ke_route_length_m = ke_route_metric.length
ke_footprint_metric = ke_route_metric.buffer(selected_beam_m / 2, cap_style=1, join_style=1)

# Densely sample the centerline, expand by one H3 ring, then retain hexes touched
# by the swept circular footprint. This avoids centroid-only polyfill omissions.
sample_distances = np.arange(0.0, ke_route_length_m + 25.0, 25.0)
sample_points_metric = gpd.GeoSeries(
    [ke_route_metric.interpolate(min(distance, ke_route_length_m)) for distance in sample_distances],
    crs=32610,
).to_crs(4326)
centerline_cells = {
    h3.latlng_to_cell(point.y, point.x, KE_H3_RESOLUTION)
    for point in sample_points_metric
}
candidate_cells = set().union(*(h3.grid_disk(cell, 1) for cell in centerline_cells))

def h3_polygon(cell: str) -> Polygon:
    return Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(cell)])

ke_hexes = gpd.GeoDataFrame(
    {"h3": sorted(candidate_cells)},
    geometry=[h3_polygon(cell) for cell in sorted(candidate_cells)],
    crs=4326,
)
ke_hexes_metric = ke_hexes.to_crs(32610)
ke_hexes_metric["footprint_touched"] = ke_hexes_metric.intersects(ke_footprint_metric)
ke_hexes_metric = ke_hexes_metric.loc[ke_hexes_metric["footprint_touched"]].copy()
ke_hexes_metric["route_intersection"] = ke_hexes_metric.geometry.intersection(ke_route_metric)
ke_hexes_metric["transect_distance_m"] = ke_hexes_metric["route_intersection"].length

def intersection_progress_bounds(intersection):
    if intersection.is_empty:
        return (np.nan, np.nan)
    lines = list(intersection.geoms) if intersection.geom_type == "MultiLineString" else [intersection]
    progress = []
    for line in lines:
        if line.geom_type == "LineString" and len(line.coords) >= 2:
            progress.extend([ke_route_metric.project(Point(line.coords[0])), ke_route_metric.project(Point(line.coords[-1]))])
    return (min(progress), max(progress)) if progress else (np.nan, np.nan)

progress_bounds = ke_hexes_metric["route_intersection"].map(intersection_progress_bounds)
ke_hexes_metric["route_start_m"] = progress_bounds.map(lambda bounds: bounds[0])
ke_hexes_metric["route_end_m"] = progress_bounds.map(lambda bounds: bounds[1])
ke_hexes_metric["distance_fraction"] = ke_hexes_metric["transect_distance_m"] / ke_route_length_m
ke_hexes_metric["voyage_time_in_grid_min"] = KE_VOYAGE_DURATION_MIN * ke_hexes_metric["distance_fraction"]

positive_distance = ke_hexes_metric["transect_distance_m"] > 1e-6
distance_error_m = abs(ke_hexes_metric.loc[positive_distance, "transect_distance_m"].sum() - ke_route_length_m)
time_error_min = abs(ke_hexes_metric["voyage_time_in_grid_min"].sum() - KE_VOYAGE_DURATION_MIN)
if distance_error_m > 1.0 or time_error_min > 1e-6:
    raise ValueError(f"H3 route partition failed conservation: {distance_error_m=:.3f}, {time_error_min=:.3g}")

# Allocate every voyage on the selected date along the same route partition.
ke_day_voyages = ke_ridership.loc[ke_ridership["service_date"].eq(KE_SELECTED_DATE)].copy()
selected_day_total_riders = ke_day_voyages["reported_total_riders"].sum()
ke_hexes_metric["allocated_daily_ridership"] = selected_day_total_riders * ke_hexes_metric["distance_fraction"]
ke_hexes_metric["daily_voyage_count"] = len(ke_day_voyages)
ke_hexes_metric["daily_ferry_time_min"] = len(ke_day_voyages) * ke_hexes_metric["voyage_time_in_grid_min"]
ke_hexes_metric["average_hourly_time_in_grid_min"] = ke_hexes_metric["daily_ferry_time_min"] / 24.0
if abs(ke_hexes_metric["allocated_daily_ridership"].sum() - selected_day_total_riders) > 1e-6:
    raise ValueError("Daily route ridership was not conserved across H3 cells.")

ke_hex_grid = ke_hexes_metric.drop(columns=["route_intersection"]).to_crs(4326)
ke_hex_grid["h3_resolution"] = KE_H3_RESOLUTION
ke_hex_grid["selected_date"] = KE_SELECTED_DATE
ke_hex_grid["selected_sailing_id"] = selected_voyage["sailing_id"]
ke_hex_grid["route_id"] = ke_route.route_id
ke_hex_grid["route_name"] = ke_route.route_name

In [ ]:
# Split each voyage's cell interval at clock-hour boundaries. This produces exact
# ferry-minutes and rider-minutes per H3 cell and clock hour for the selected date.
hourly_records = []
route_cells = ke_hexes_metric.loc[positive_distance].copy()
for voyage in ke_day_voyages.itertuples(index=False):
    forward = voyage.origin_terminal == KE_SELECTED_ORIGIN
    riders_on_voyage = float(voyage.reported_total_riders)
    for cell in route_cells.itertuples(index=False):
        if forward:
            offset_start = KE_VOYAGE_DURATION_MIN * cell.route_start_m / ke_route_length_m
            offset_end = KE_VOYAGE_DURATION_MIN * cell.route_end_m / ke_route_length_m
        else:
            offset_start = KE_VOYAGE_DURATION_MIN * (1.0 - cell.route_end_m / ke_route_length_m)
            offset_end = KE_VOYAGE_DURATION_MIN * (1.0 - cell.route_start_m / ke_route_length_m)
        interval_start = voyage.departure_local + pd.to_timedelta(offset_start, unit="m")
        interval_end = voyage.departure_local + pd.to_timedelta(offset_end, unit="m")
        cursor = interval_start
        while cursor < interval_end:
            hour_start = cursor.floor("h")
            piece_end = min(interval_end, hour_start + pd.Timedelta(hours=1))
            ferry_minutes = (piece_end - cursor).total_seconds() / 60.0
            hourly_records.append(
                {
                    "hour_start": hour_start, "h3": cell.h3, "sailing_id": voyage.sailing_id,
                    "origin_terminal": voyage.origin_terminal, "destination_terminal": voyage.destination_terminal,
                    "ferry_minutes": ferry_minutes, "rider_minutes": riders_on_voyage * ferry_minutes,
                }
            )
            cursor = piece_end

ke_hourly_grid = (
    pd.DataFrame(hourly_records)
    .groupby(["hour_start", "h3"], as_index=False)
    .agg(
        ferry_minutes=("ferry_minutes", "sum"),
        voyages_touching=("sailing_id", "nunique"),
        rider_minutes=("rider_minutes", "sum"),
    )
)
ke_hourly_grid["mean_time_in_grid_per_voyage_min"] = (
    ke_hourly_grid["ferry_minutes"] / ke_hourly_grid["voyages_touching"]
)
ke_hourly_grid["rider_hours"] = ke_hourly_grid["rider_minutes"] / 60.0

grid_display_columns = [
    "h3", "footprint_touched", "transect_distance_m", "distance_fraction",
    "voyage_time_in_grid_min", "average_hourly_time_in_grid_min", "allocated_daily_ridership",
]
display(
    ke_hex_grid.sort_values("route_start_m")[grid_display_columns]
    .style.format(
        {"transect_distance_m": "{:,.1f}", "distance_fraction": "{:.4f}",
         "voyage_time_in_grid_min": "{:.2f}", "average_hourly_time_in_grid_min": "{:.2f}",
         "allocated_daily_ridership": "{:,.1f}"}
    )
)
display(ke_hourly_grid.sort_values(["hour_start", "h3"]).head(30))
print(
    f"Selected {len(ke_hex_grid):,} footprint-touched H3 cells; "
    f"{positive_distance.sum():,} contain centerline distance. "
    f"The {len(ke_day_voyages):,} voyages on {KE_SELECTED_DATE:%Y-%m-%d} carried "
    f"{selected_day_total_riders:,.0f} reported riders."
)

In [ ]:
KE_GRID_GEOJSON = KE_OUTPUT_DIR / "kingston_edmonds_h3_grid.geojson"
KE_GRID_PARQUET = KE_OUTPUT_DIR / "kingston_edmonds_h3_grid.parquet"
KE_HOURLY_PARQUET = KE_OUTPUT_DIR / "kingston_edmonds_hourly_time_in_grid.parquet"
KE_GRID_MAP = KE_OUTPUT_DIR / "kingston_edmonds_h3_time_in_grid_map.html"
ke_hex_grid.drop(columns=["selected_date"]).to_file(KE_GRID_GEOJSON, driver="GeoJSON")
ke_hex_grid.to_parquet(KE_GRID_PARQUET, index=False)
ke_hourly_grid.to_parquet(KE_HOURLY_PARQUET, index=False)

map_hex_grid = ke_hex_grid.drop(columns=["selected_date"])
hex_geojson = json.loads(map_hex_grid.to_json())
customdata = np.column_stack(
    [
        ke_hex_grid["transect_distance_m"], ke_hex_grid["distance_fraction"],
        ke_hex_grid["voyage_time_in_grid_min"], ke_hex_grid["average_hourly_time_in_grid_min"],
        ke_hex_grid["allocated_daily_ridership"],
    ]
)
fig = go.Figure(
    go.Choroplethmap(
        geojson=hex_geojson, locations=ke_hex_grid["h3"], featureidkey="properties.h3",
        z=ke_hex_grid["voyage_time_in_grid_min"], colorscale="Viridis", marker_opacity=0.58,
        marker_line_width=1.2, marker_line_color="#17202A", customdata=customdata,
        colorbar={"title": "Minutes in cell<br>per voyage"},
        hovertemplate=(
            "<b>%{location}</b><br>Transect distance: %{customdata[0]:,.1f} m<br>"
            "Route fraction: %{customdata[1]:.3%}<br>Time per voyage: %{customdata[2]:.2f} min<br>"
            "Average ferry time per clock hour: %{customdata[3]:.2f} min<br>"
            "Allocated daily ridership: %{customdata[4]:,.1f}<extra></extra>"
        ),
    )
)
ke_footprint_wgs84 = gpd.GeoSeries([ke_footprint_metric], crs=32610).to_crs(4326).iloc[0]
footprint_lon, footprint_lat = ke_footprint_wgs84.exterior.xy
fig.add_trace(
    go.Scattermap(
        lon=list(footprint_lon), lat=list(footprint_lat), mode="lines",
        line={"width": 2, "color": "#59A14F"}, name=f"{selected_beam_ft:.0f}-ft swept footprint",
        hovertemplate=f"Swept circular footprint ({selected_beam_ft:.0f}-ft diameter)<extra></extra>",
    )
)
route_lon, route_lat = ke_oriented_wgs84.xy
fig.add_trace(
    go.Scattermap(
        lon=list(route_lon), lat=list(route_lat), mode="lines",
        line={"width": 4, "color": "#E15759"}, name="Kingston → Edmonds centerline",
        hovertemplate="Kingston → Edmonds route centerline<extra></extra>",
    )
)
map_bounds = ke_hex_grid.total_bounds
fig.update_layout(
    title=(
        f"Kingston → Edmonds selected H3 cells and time-in-grid<br>"
        f"<sup>{selected_voyage['departure_local']:%Y-%m-%d %H:%M}, {selected_voyage['vessel_name']}, "
        f"{KE_VOYAGE_DURATION_MIN:.0f}-minute voyage, {selected_beam_ft:.0f}-ft beam, H3 r{KE_H3_RESOLUTION}</sup>"
    ),
    map={"style": "open-street-map", "center": {"lon": float((map_bounds[0] + map_bounds[2]) / 2), "lat": float((map_bounds[1] + map_bounds[3]) / 2)}, "zoom": 9.3},
    height=700, margin={"l": 0, "r": 0, "t": 80, "b": 0}, hoverlabel={"align": "left"},
)
fig.write_html(KE_GRID_MAP, include_plotlyjs=True, full_html=True)
print(f"Saved selected H3 grid to {KE_GRID_GEOJSON} and {KE_GRID_PARQUET}")
print(f"Saved exact hourly time-in-grid table to {KE_HOURLY_PARQUET}")
print(f"Saved interactive diagnostic map to {KE_GRID_MAP}")
fig.show()

## Hourly ferry-time clock map

Use the slider or play button to move through the selected service date. Color represents the total number of ferry-minutes spent in each H3 cell during that clock hour, including exact splits when a voyage crosses an hour boundary.

In [ ]:
KE_HOURLY_CLOCK_PARQUET = KE_OUTPUT_DIR / "kingston_edmonds_hourly_clock_by_cell.parquet"
KE_HOURLY_CLOCK_MAP = KE_OUTPUT_DIR / "kingston_edmonds_hourly_ferry_time_clock_map.html"

service_hours = pd.date_range(
    ke_hourly_grid["hour_start"].min(),
    ke_hourly_grid["hour_start"].max(),
    freq="h",
)
clock_grid = pd.MultiIndex.from_product(
    [service_hours, ke_hex_grid["h3"]], names=["hour_start", "h3"]
).to_frame(index=False)
clock_grid = clock_grid.merge(ke_hourly_grid, on=["hour_start", "h3"], how="left")
clock_metrics = ["ferry_minutes", "voyages_touching", "rider_minutes", "mean_time_in_grid_per_voyage_min", "rider_hours"]
clock_grid[clock_metrics] = clock_grid[clock_metrics].fillna(0.0)
clock_grid["clock_label"] = clock_grid["hour_start"].dt.strftime("%b %d, %H:%M")
clock_grid.to_parquet(KE_HOURLY_CLOCK_PARQUET, index=False)

clock_zmax = max(float(clock_grid["ferry_minutes"].max()), 1.0)

def clock_hex_trace(hour_frame: pd.DataFrame, show_scale: bool = True):
    hour_frame = hour_frame.set_index("h3").reindex(ke_hex_grid["h3"]).reset_index()
    hour_customdata = np.column_stack(
        [
            hour_frame["clock_label"], hour_frame["voyages_touching"],
            hour_frame["mean_time_in_grid_per_voyage_min"], hour_frame["rider_hours"],
        ]
    )
    return go.Choroplethmap(
        geojson=hex_geojson, locations=hour_frame["h3"], featureidkey="properties.h3",
        z=hour_frame["ferry_minutes"], zmin=0, zmax=clock_zmax, colorscale="Turbo",
        marker_opacity=0.66, marker_line_width=1.2, marker_line_color="#17202A",
        customdata=hour_customdata, showscale=show_scale,
        colorbar={"title": "Ferry-minutes<br>in clock hour"},
        hovertemplate=(
            "<b>%{location}</b><br>Clock hour: %{customdata[0]}<br>"
            "Ferry-minutes: %{z:.2f}<br>Voyages touching cell: %{customdata[1]:.0f}<br>"
            "Mean minutes per touching voyage: %{customdata[2]:.2f}<br>"
            "Rider-hours: %{customdata[3]:,.2f}<extra></extra>"
        ),
    )

clock_frames = []
for hour_start, hour_frame in clock_grid.groupby("hour_start", sort=True):
    label = pd.Timestamp(hour_start).strftime("%b %d, %H:%M")
    clock_frames.append(
        go.Frame(
            name=label, data=[clock_hex_trace(hour_frame, show_scale=True)], traces=[0],
            layout=go.Layout(
                title={"text": f"Kingston–Edmonds ferry time by H3 cell<br><sup>Clock hour beginning {label}</sup>"}
            ),
        )
    )

first_hour = service_hours[0]
first_frame = clock_grid.loc[clock_grid["hour_start"].eq(first_hour)]
clock_fig = go.Figure(data=[clock_hex_trace(first_frame)], frames=clock_frames)
clock_fig.add_trace(
    go.Scattermap(
        lon=list(footprint_lon), lat=list(footprint_lat), mode="lines",
        line={"width": 2, "color": "#59A14F"}, name=f"{selected_beam_ft:.0f}-ft swept footprint",
        hovertemplate=f"Swept circular footprint ({selected_beam_ft:.0f}-ft diameter)<extra></extra>",
    )
)
clock_fig.add_trace(
    go.Scattermap(
        lon=list(route_lon), lat=list(route_lat), mode="lines",
        line={"width": 4, "color": "#E15759"}, name="Kingston → Edmonds centerline",
        hovertemplate="Kingston–Edmonds route centerline<extra></extra>",
    )
)
clock_fig.add_trace(
    go.Scattermap(
        lon=[route_coordinates[0][0], route_coordinates[-1][0]],
        lat=[route_coordinates[0][1], route_coordinates[-1][1]],
        mode="markers+text", marker={"size": 10, "color": "#FFFFFF"},
        text=[KE_SELECTED_ORIGIN, KE_SELECTED_DESTINATION], textposition="top center",
        name="Terminals", hovertemplate="%{text}<extra></extra>",
    )
)
slider_steps = [
    {
        "label": frame.name, "method": "animate",
        "args": [[frame.name], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
    }
    for frame in clock_frames
]
clock_fig.update_layout(
    title={"text": f"Kingston–Edmonds ferry time by H3 cell<br><sup>Clock hour beginning {first_hour:%b %d, %H:%M}</sup>"},
    map={"style": "open-street-map", "center": {"lon": float((map_bounds[0] + map_bounds[2]) / 2), "lat": float((map_bounds[1] + map_bounds[3]) / 2)}, "zoom": 9.3},
    sliders=[{"active": 0, "currentvalue": {"prefix": "Clock hour: "}, "pad": {"t": 45}, "steps": slider_steps}],
    updatemenus=[
        {
            "type": "buttons", "direction": "left", "x": 0.01, "y": 0.02,
            "buttons": [
                {"label": "Play", "method": "animate", "args": [None, {"fromcurrent": True, "frame": {"duration": 900, "redraw": True}, "transition": {"duration": 200}}]},
                {"label": "Pause", "method": "animate", "args": [[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}, "transition": {"duration": 0}}]},
            ],
        }
    ],
    height=760, margin={"l": 0, "r": 0, "t": 80, "b": 125}, hoverlabel={"align": "left"},
)
clock_fig.write_html(KE_HOURLY_CLOCK_MAP, include_plotlyjs=True, full_html=True)
print(f"Saved hourly clock table to {KE_HOURLY_CLOCK_PARQUET}")
print(f"Saved animated hourly ferry-time map to {KE_HOURLY_CLOCK_MAP}")
clock_fig.show()

## Summed ferry exposure by grid and date

The production pipeline combines both directions and all vessels into raw daily source-cell exposure. It uses centerline-intersected cells only, preserves rider- and vessel-minute conservation, and writes both route-level and summed date × H3 products without normalization. The selected Kingston–Edmonds date remains a QA example below.

In [ ]:
import sys

FERRY_UTILS_DIR = REPO_ROOT / "notebooks/data/ferry_utils"
if str(FERRY_UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(FERRY_UTILS_DIR))

from human.activity_and_effort.ferry.pipeline import build_ferry_daily_source_weights

FERRY_SOURCE_OUTPUT_DIR = FERRY_UTILS_DIR / "outputs/ferry_source_weights_daily"
WSF_VESSEL_HISTORY = FERRY_UTILS_DIR / "outputs/wsf_vessel_history/wsf_vessel_history_2025-07-20.parquet"
ferry_source_result = build_ferry_daily_source_weights(
    wsf_ridership_path=FERRY_UTILS_DIR / "wsf_ridership.parquet",
    wsf_vessel_history_path=WSF_VESSEL_HISTORY if WSF_VESSEL_HISTORY.exists() else None,
    bc_ridership_path=FERRY_UTILS_DIR / "bc_ferries_estimated_ridership.parquet",
    route_segments_path=FERRY_UTILS_DIR / "outputs/ferry_routes/wa_bc_ferry_route_segments.parquet",
    route_config_path=FERRY_UTILS_DIR / "ferry_route_effort.yaml",
    output_dir=FERRY_SOURCE_OUTPUT_DIR,
    h3_resolution=KE_H3_RESOLUTION,
    allow_incomplete_routes=True,  # unresolved routes are written to the metadata/report
)

ferry_daily_by_route = pd.read_parquet(ferry_source_result.route_level_path)
ke_production_daily = ferry_daily_by_route.loc[
    ferry_daily_by_route["route_key"].eq("Edmonds__Kingston")
    & ferry_daily_by_route["service_date"].eq(KE_SELECTED_DATE)
].sort_values("source_h3")
qa_columns = [
    "source_h3", "daily_riders", "daily_sailings", "distance_fraction",
    "voyage_time_in_cell_minutes", "ferry_rider_minutes",
    "ferry_rider_hours", "ferry_vessel_minutes",
    "route_duration_minutes", "voyage_duration_source",
    "voyage_duration_history_coverage_fraction",
]
display(ke_production_daily[qa_columns])
print(f"Saved route-level daily source weights to {ferry_source_result.route_level_path}")
print(f"Saved summed date × H3 source weights to {ferry_source_result.collapsed_path}")

## All-route daily rider-minute map — July 20, 2025

This date-specific QA map includes every WSF route key present in the ridership artifact on July 20, 2025. H3 cells are colored by rider-minutes summed across routes. Route hover reports the operational voyage duration inferred from WSDOT `ActualDepart` to `EstArrival`, API-history coverage, configured fallbacks, ridership, sailing count, and traversal assumptions. This is an estimated operational duration because the API does not provide an actual arrival timestamp. BC Ferries has no records on this date because the current BC artifact begins in 2026.

In [ ]:
from IPython.display import IFrame
from map_ferry_source_weights_for_date import build_date_map

ALL_ROUTES_MAP_DATE = pd.Timestamp("2025-07-20")
ALL_ROUTES_MAP_HTML = (
    FERRY_SOURCE_OUTPUT_DIR / "maps/ferry_source_weights_2025-07-20.html"
)
all_routes_map, all_routes_cells, all_routes_lines = build_date_map(
    route_level_path=ferry_source_result.route_level_path,
    route_segments_path=FERRY_UTILS_DIR / "outputs/ferry_routes/wa_bc_ferry_route_segments.parquet",
    route_config_path=FERRY_UTILS_DIR / "ferry_route_effort.yaml",
    service_date=ALL_ROUTES_MAP_DATE,
    output_html=ALL_ROUTES_MAP_HTML,
)
print(f"Saved date-cell subset to {all_routes_cells}")
print(f"Saved active route lines to {all_routes_lines}")
IFrame(src=all_routes_map.as_uri(), width=1100, height=780)